# LeetCode 1746 — Maximum Subarray Sum After One Operation bằng Python

Link đề: https://leetcode.com/problems/maximum-subarray-sum-after-one-operation/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming (Kadane mở rộng)

> 🔒 **Bài Premium.** Đề trong notebook được viết lại theo hiểu biết về bài (nguồn: kho lời giải doocs/leetcode). Nếu bạn có tài khoản Premium, nên đối chiếu lại câu chữ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def maxSumAfterOperation(self, nums: List[int]) -> int:
        ...
```

Lộ trình: **vét cạn `O(n³)`** (chọn chỉ số bình phương + mọi mảng con) → **mỗi chỉ số chạy Kadane `O(n²)`** → **Kadane trái + Kadane phải `O(n)`** → **DP hai trạng thái `O(1)` bộ nhớ** (lời giải nộp).

## 1. Mục tiêu học

1. Ôn lại **thuật toán Kadane** (mảng con tổng lớn nhất) dưới dạng DP "kết thúc tại `i`".
2. Mở rộng Kadane bằng **thêm một chiều trạng thái**: "đã dùng thao tác hay chưa" — kỹ thuật DP rất hay gặp (bài 1186 *Maximum Subarray Sum with One Deletion* cùng khuôn).
3. Viết đúng **thứ tự cập nhật** khi nhiều biến trạng thái phụ thuộc nhau (dùng biến tạm hoặc gán song song).
4. Dùng kỹ thuật **tiền tố/hậu tố** (best ending at `i-1`, best starting at `i+1`) để "ghép" lời giải quanh một phần tử đặc biệt.
5. Kiểm chứng DP bằng vét cạn ngẫu nhiên có seed, và chứng minh một chiến lược tham lam trực giác là **sai** bằng phản ví dụ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Bạn **bắt buộc** thực hiện **đúng một** thao tác: chọn một phần tử `nums[i]` và thay nó bằng `nums[i] * nums[i]`.

Trả về **tổng lớn nhất** của một mảng con **khác rỗng** (đoạn liên tiếp) sau đúng một thao tác.

**Ví dụ 1:** `nums = [2,-1,-4,-3]` → `17`. Bình phương chỉ số 2: `[2,-1,16,-3]`; mảng con `2 + (-1) + 16 = 17`.

**Ví dụ 2:** `nums = [1,-1,1,1,-1,-1,1]` → `4`. Bình phương chỉ số 1: `[1,1,1,1,-1,-1,1]`; mảng con `1+1+1+1 = 4`.

**Ràng buộc:**
- `1 ≤ nums.length ≤ 10⁵`
- `-10⁴ ≤ nums[i] ≤ 10⁴`

In [1]:
import random
import time
from math import inf
from typing import List

examples = [([2, -1, -4, -3], 17), ([1, -1, 1, 1, -1, -1, 1], 4)]
for nums, want in examples:
    print(nums, "-> kỳ vọng", want)

[2, -1, -4, -3] -> kỳ vọng 17
[1, -1, 1, 1, -1, -1, 1] -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1 Mảng con (subarray) và tổng tiền tố

Mảng con là đoạn **liên tiếp** `nums[l..r]`. Có `n(n+1)/2` mảng con khác rỗng. Tổng tiền tố `P[k] = nums[0] + … + nums[k-1]` cho tổng đoạn `P[r+1] - P[l]` trong `O(1)`.

In [2]:
nums = [2, -1, 16, -3]
P = [0]
for x in nums:
    P.append(P[-1] + x)
print("P =", P)
print("tổng nums[0..2] =", P[3] - P[0])
print("số mảng con khác rỗng:", len(nums) * (len(nums) + 1) // 2)

P = [0, 2, 1, 17, 14]
tổng nums[0..2] = 17
số mảng con khác rỗng: 10


### 3.2 Kadane: mảng con tổng lớn nhất trong `O(n)`

`cur` = tổng lớn nhất của mảng con **kết thúc tại** vị trí hiện tại. Với phần tử `x`: hoặc nối vào đoạn trước (`cur + x`), hoặc bắt đầu mới (`x`) ⇒ `cur = max(cur, 0) + x`. Đáp án là max của mọi `cur`.

In [3]:
def kadane(a):
    cur, best = 0, -inf
    for x in a:
        cur = max(cur, 0) + x
        best = max(best, cur)
    return best

print(kadane([2, -1, 16, -3]), kadane([-3, -1, -2]), kadane([1, 1, 1, 1, -1, -1, 1]))

17 -1 4


### 3.3 Bình phương luôn không nhỏ hơn số ban đầu (với số nguyên)

Với số nguyên `x`: nếu `x ≤ 0` thì `x² ≥ 0 ≥ x`; nếu `x ≥ 1` thì `x² ≥ x`. Vậy **bình phương không bao giờ làm giảm** giá trị. Hệ quả: thao tác "bắt buộc" không gây hại, và đáp án luôn ≥ `max(x²) ≥ 0`.

In [4]:
print(all(x * x >= x for x in range(-10**4, 10**4 + 1)))

True


### 3.4 Gán song song trong Python

`f, g = A, B` tính **cả hai vế phải trước** rồi mới gán. Rất quan trọng khi `B` phụ thuộc giá trị **cũ** của `f`.

In [5]:
f, g = 1, 10
f, g = f + 100, f * 2      # g dùng f CŨ = 1
print(f, g)
f, g = 1, 10
f = f + 100
g = f * 2                  # g dùng f MỚI = 101
print(f, g)

101 2
101 202


### 3.5 `math.inf` làm giá trị khởi tạo cho max

`-inf` nhỏ hơn mọi số nên `max(-inf, x) = x`. Tránh khởi tạo `best = 0` khi đáp án có thể âm (với Kadane thường); ở bài này đáp án luôn ≥ 0 nhưng dùng `-inf` vẫn là thói quen an toàn.

In [6]:
print(max(-inf, -5), -inf < -10**18)

-5 True


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 `nums = [2,-1,-4,-3]`. Thử bình phương từng chỉ số rồi chạy Kadane bằng tay:

| bình phương chỉ số | mảng mới | mảng con tốt nhất | tổng |
|---|---|---|---|
| 0 | `[4,-1,-4,-3]` | `[4]` | 4 |
| 1 | `[2,1,-4,-3]` | `[2,1]` | 3 |
| 2 | `[2,-1,16,-3]` | `[2,-1,16]` | **17** |
| 3 | `[2,-1,-4,9]` | `[9]` | 9 |

Quan sát: mảng con tối ưu **chứa** phần tử bị bình phương (nếu không chứa thì thao tác vô ích — đã thấy ở 3.3 bình phương không hại, nên luôn có lời giải tối ưu chứa nó). Mảng con đó = **(đoạn kết thúc ngay trước `i`, có thể rỗng)** + `nums[i]²` + **(đoạn bắt đầu ngay sau `i`, có thể rỗng)**. Hai đoạn bên chỉ nên lấy khi tổng của chúng dương.

In [7]:
nums = [2, -1, -4, -3]
for i in range(len(nums)):
    b = nums[:]; b[i] = b[i] ** 2
    print(f"bình phương i={i}: {b} -> Kadane = {kadane(b)}")

bình phương i=0: [4, -1, -4, -3] -> Kadane = 4
bình phương i=1: [2, 1, -4, -3] -> Kadane = 3
bình phương i=2: [2, -1, 16, -3] -> Kadane = 17
bình phương i=3: [2, -1, -4, 9] -> Kadane = 9


## 5. Cách 1 — Brute Force / baseline: chọn `i` và mọi mảng con (`O(n³)`)

**Ý tưởng:** duyệt mọi chỉ số `i` để bình phương, mọi điểm đầu `l`, mở rộng điểm cuối `r` và cộng dồn tổng (thay `nums[i]` bằng `nums[i]²` khi `r == i`). Không cần lập luận gì — chắc chắn đúng, dùng làm chuẩn.

In [8]:
def max_after_op_brute(nums):
    n = len(nums)
    best = -inf
    for i in range(n):
        for l in range(n):
            s = 0
            for r in range(l, n):
                s += nums[r] * nums[r] if r == i else nums[r]
                best = max(best, s)
    return best

for nums, want in examples:
    print(nums, "->", max_after_op_brute(nums), "| kỳ vọng", want)

[2, -1, -4, -3] -> 17 | kỳ vọng 17
[1, -1, 1, 1, -1, -1, 1] -> 4 | kỳ vọng 4


**Độ phức tạp:** `O(n³)` thời gian, `O(1)` bộ nhớ. `n = 10⁵` ⇒ `10¹⁵` phép — bất khả.

**Nút thắt:** với mỗi `i`, ta tính lại mọi mảng con từ đầu. Nhưng "mảng con tốt nhất" chính là việc Kadane làm trong `O(n)`.

In [9]:
rng = random.Random(0)
arr = [rng.randint(-10, 10) for _ in range(120)]
t = time.perf_counter(); r = max_after_op_brute(arr); dt = time.perf_counter() - t
print(f"n=120: {r} trong {dt:.3f}s -> n=10^5 ước tính ≈ {dt * (10**5 / 120) ** 3 / 3600 / 24 / 365:.0f} năm")

n=120: 185 trong 0.034s -> n=10^5 ước tính ≈ 1 năm


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — chỉ cần mảng con chứa phần tử bị bình phương.** Vì `x² ≥ x`, mọi mảng con không chứa `i` đều có tổng ≤ cùng mảng con đó sau khi chuyển thao tác vào một phần tử bên trong nó.

**Insight 2 — tách quanh `i`:** tổng tốt nhất khi bình phương `i` là
`L[i] + nums[i]² + R[i]`, với `L[i]` = tổng lớn nhất của đoạn **kết thúc tại `i-1`** (cho phép rỗng ⇒ ≥ 0), `R[i]` = tổng lớn nhất của đoạn **bắt đầu tại `i+1`** (≥ 0). Cả hai tính bằng Kadane xuôi/ngược.

**Insight 3 — DP hai trạng thái (gọn nhất):** duyệt trái sang phải, tại mỗi vị trí giữ
- `f` = tổng lớn nhất của mảng con kết thúc tại đây, **chưa** dùng thao tác (Kadane thường);
- `g` = tổng lớn nhất của mảng con kết thúc tại đây, **đã** dùng đúng một thao tác.

Quy tắc chuyển với phần tử `x`:
- `f_mới = max(f, 0) + x` (nối hoặc bắt đầu mới).
- `g_mới = max( max(f, 0) + x²,  g + x )`: *nếu* dùng thao tác **tại chính `x`** *thì* nối sau đoạn chưa-dùng tốt nhất (hoặc bắt đầu mới); *nếu* đã dùng **trước đó** *thì* chỉ nối `x` vào.

Đáp án = max của mọi `g` (và `f`, dù `g ≥ f` luôn đúng).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Thử từng chỉ số, mỗi lần chạy Kadane (`O(n²)`)

In [10]:
def max_after_op_each(nums):
    best = -inf
    for i in range(len(nums)):
        cur = 0
        for j, x in enumerate(nums):
            v = x * x if j == i else x
            cur = max(cur, 0) + v
            best = max(best, cur)
    return best

for nums, want in examples:
    print(max_after_op_each(nums), want)

17 17
4 4


**Độ phức tạp:** `O(n²)` thời gian, `O(1)` bộ nhớ. `n = 10⁵` ⇒ `10¹⁰` — vẫn quá chậm. Nút thắt: mỗi lần đổi `i`, phần Kadane bên trái `i` và bên phải `i` không đổi nhưng vẫn tính lại ⇒ **tính trước một lần** cho mọi `i`.

### 7.2 Cách 3 — Kadane trái + Kadane phải (`O(n)` thời gian, `O(n)` bộ nhớ)

1. `left[i]` = Kadane kết thúc tại `i` (bình thường). Phần "đoạn kết thúc tại `i-1` có thể rỗng" = `max(left[i-1], 0)`.
2. `right[i]` = Kadane **bắt đầu** tại `i` (chạy từ phải sang trái). Phần bên phải = `max(right[i+1], 0)`.
3. Đáp án = `max_i ( max(left[i-1],0) + nums[i]² + max(right[i+1],0) )`.

In [11]:
def max_after_op_lr(nums, verbose=False):
    n = len(nums)
    left = [0] * n          # best subarray ending at i (non-empty)
    cur = 0
    for i, x in enumerate(nums):
        cur = max(cur, 0) + x
        left[i] = cur
    right = [0] * n         # best subarray starting at i (non-empty)
    cur = 0
    for i in range(n - 1, -1, -1):
        cur = max(cur, 0) + nums[i]
        right[i] = cur
    best = -inf
    for i, x in enumerate(nums):
        L = max(left[i - 1], 0) if i > 0 else 0
        R = max(right[i + 1], 0) if i + 1 < n else 0
        if verbose:
            print(f"i={i}: L={L:>3} + x²={x*x:>3} + R={R:>3} = {L + x*x + R}")
        best = max(best, L + x * x + R)
    if verbose:
        print("left =", left, "| right =", right)
    return best

print("đáp án:", max_after_op_lr([2, -1, -4, -3], verbose=True))

i=0: L=  0 + x²=  4 + R=  0 = 4
i=1: L=  2 + x²=  1 + R=  0 = 3
i=2: L=  1 + x²= 16 + R=  0 = 17
i=3: L=  0 + x²=  9 + R=  0 = 9
left = [2, 1, -3, -3] | right = [2, -1, -4, -3]
đáp án: 17


In [12]:
class SolutionLR:
    def maxSumAfterOperation(self, nums: List[int]) -> int:
        return max_after_op_lr(nums)

for nums, want in examples:
    print(SolutionLR().maxSumAfterOperation(nums), want)

17 17
4 4


**Độ phức tạp:** ba lượt duyệt ⇒ `O(n)` thời gian, hai mảng phụ ⇒ `O(n)` bộ nhớ.

### 7.3 Cách 4 — DP hai trạng thái `f`, `g` (`O(n)` thời gian, `O(1)` bộ nhớ)

Gộp hai lượt Kadane vào một lượt nhờ trạng thái "đã dùng thao tác chưa". Demo in bảng:

In [13]:
def max_after_op_dp_demo(nums):
    f = g = 0
    best = -inf
    print(f"{'x':>4} | {'f (chưa dùng)':>14} | {'g (đã dùng)':>12} | best")
    for x in nums:
        f, g = max(f, 0) + x, max(max(f, 0) + x * x, g + x)   # gán song song: g dùng f CŨ
        best = max(best, f, g)
        print(f"{x:>4} | {f:>14} | {g:>12} | {best}")
    return best

max_after_op_dp_demo([2, -1, -4, -3])

   x |  f (chưa dùng) |  g (đã dùng) | best
   2 |              2 |            4 | 4
  -1 |              1 |            3 | 4
  -4 |             -3 |           17 | 17
  -3 |             -3 |           14 | 17


17

In [14]:
class SolutionDP:
    def maxSumAfterOperation(self, nums: List[int]) -> int:
        f = g = 0
        best = -inf
        for x in nums:
            f, g = max(f, 0) + x, max(max(f, 0) + x * x, g + x)
            best = max(best, g)
        return best

for nums, want in examples:
    print(SolutionDP().maxSumAfterOperation(nums), want)

17 17
4 4


Ở phần tử đầu tiên, `g` ban đầu = 0 nên `g + x` tương ứng với "đoạn đã-dùng rỗng" — không hợp lệ về mặt ý nghĩa, nhưng không gây sai vì `max(f,0) + x² = x² ≥ x = 0 + x`. Nếu muốn chặt chẽ, có thể khởi tạo `g = -inf`.

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

## 8. TRACE từng bước

Trace Cách 4 trên Ví dụ 2: `nums = [1,-1,1,1,-1,-1,1]`.

| i | x | `max(f,0)` cũ | `f` mới | `max(f,0)+x²` | `g+x` | `g` mới | best |
|---|---|---|---|---|---|---|---|
| 0 | 1 | 0 | 1 | 1 | 1 | 1 | 1 |
| 1 | -1 | 1 | 0 | 2 | 0 | 2 | 2 |
| 2 | 1 | 0 | 1 | 1 | 3 | 3 | 3 |
| 3 | 1 | 1 | 2 | 2 | 4 | 4 | 4 |
| 4 | -1 | 2 | 1 | 3 | 3 | 3 | 4 |
| 5 | -1 | 1 | 0 | 2 | 2 | 2 | 4 |
| 6 | 1 | 0 | 1 | 1 | 3 | 3 | 4 |

Đáp án 4: đạt ở `i = 3` với `g` = đoạn `[1, (-1)²=1, 1, 1]`.

In [15]:
def trace_dp(nums):
    f = g = 0
    best = -inf
    for i, x in enumerate(nums):
        pf = max(f, 0)
        use_here = pf + x * x
        use_before = g + x
        f, g = pf + x, max(use_here, use_before)
        best = max(best, g)
        src = "dùng tại x" if use_here >= use_before else "đã dùng trước"
        print(f"i={i} x={x:>2} | max(f,0)={pf} | f={f:>2} | dùng tại đây={use_here:>2}, đã dùng trước={use_before:>2}"
              f" -> g={g:>2} ({src}) | best={best}")
    return best

trace_dp([1, -1, 1, 1, -1, -1, 1])

i=0 x= 1 | max(f,0)=0 | f= 1 | dùng tại đây= 1, đã dùng trước= 1 -> g= 1 (dùng tại x) | best=1
i=1 x=-1 | max(f,0)=1 | f= 0 | dùng tại đây= 2, đã dùng trước= 0 -> g= 2 (dùng tại x) | best=2
i=2 x= 1 | max(f,0)=0 | f= 1 | dùng tại đây= 1, đã dùng trước= 3 -> g= 3 (đã dùng trước) | best=3
i=3 x= 1 | max(f,0)=1 | f= 2 | dùng tại đây= 2, đã dùng trước= 4 -> g= 4 (đã dùng trước) | best=4
i=4 x=-1 | max(f,0)=2 | f= 1 | dùng tại đây= 3, đã dùng trước= 3 -> g= 3 (dùng tại x) | best=4
i=5 x=-1 | max(f,0)=1 | f= 0 | dùng tại đây= 2, đã dùng trước= 2 -> g= 2 (dùng tại x) | best=4
i=6 x= 1 | max(f,0)=0 | f= 1 | dùng tại đây= 1, đã dùng trước= 3 -> g= 3 (đã dùng trước) | best=4


4

Trace để **khôi phục** mảng con tối ưu (lưu thêm điểm bắt đầu và vị trí bình phương):

In [16]:
def dp_with_witness(nums):
    f, fs = 0, 0                 # f và điểm bắt đầu của đoạn f
    g, gs, gi = -inf, 0, -1      # g, điểm bắt đầu, chỉ số bình phương
    best = (-inf, 0, 0, -1)
    for i, x in enumerate(nums):
        base, base_s = (f, fs) if f > 0 else (0, i)
        cand1 = (base + x * x, base_s, i)
        cand2 = (g + x, gs, gi)
        nf, nfs = base + x, base_s
        g, gs, gi = max(cand1, cand2, key=lambda c: c[0])
        f, fs = nf, nfs
        if g > best[0]:
            best = (g, gs, i, gi)
    return best

val, l, r, sq = dp_with_witness([2, -1, -4, -3])
print(f"tổng={val}, đoạn [{l}..{r}], bình phương chỉ số {sq}")

tổng=17, đoạn [0..2], bình phương chỉ số 2


## 9. Code nộp LeetCode

In [17]:
from math import inf
from typing import List

class Solution:
    def maxSumAfterOperation(self, nums: List[int]) -> int:
        f = 0          # tốt nhất kết thúc tại đây, CHƯA bình phương
        g = 0          # tốt nhất kết thúc tại đây, ĐÃ bình phương đúng 1 phần tử
        ans = -inf
        for x in nums:
            f, g = max(f, 0) + x, max(max(f, 0) + x * x, g + x)
            ans = max(ans, g)
        return ans

for nums, want in examples:
    assert Solution().maxSumAfterOperation(nums) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Định nghĩa chính xác với mỗi chỉ số `j`:
- `F(j)` = max tổng các mảng con `nums[l..j]` (khác rỗng, không thao tác).
- `G(j)` = max tổng các mảng con `nums[l..j]` sau khi bình phương **đúng một** chỉ số `k ∈ [l, j]`.

**Công thức truy hồi:**
- `F(j) = max(F(j-1), 0) + x_j` — Kadane: hoặc nối vào đoạn tốt nhất kết thúc ở `j-1`, hoặc bắt đầu tại `j`.
- `G(j)`: xét chỉ số bị bình phương `k`.
  - `k = j`: phần trước `nums[l..j-1]` không thao tác, có thể rỗng ⇒ tốt nhất là `max(F(j-1), 0)`; cộng `x_j²`.
  - `k < j`: đoạn `nums[l..j-1]` đã chứa `k` ⇒ tốt nhất là `G(j-1)`; cộng `x_j`.
  - Hai trường hợp phủ hết và rời nhau ⇒ `G(j) = max(max(F(j-1),0) + x_j², G(j-1) + x_j)`.

**Đáp án** = `max_j G(j)`: mọi mảng con sau thao tác hoặc chứa chỉ số bị bình phương (thuộc một `G(j)`), hoặc không chứa — khi đó tổng của nó ≤ tổng của chính nó với một phần tử bên trong bị bình phương (vì `x² ≥ x`), cũng thuộc một `G(j)`.

Với cơ sở `F(-1) = 0` (rỗng) và `G(-1)` = `0` hoặc `-inf`: như đã giải thích ở 7.3, giá trị `0` không làm sai kết quả vì `x² ≥ x`.

Kiểm chứng từng mệnh đề trên bằng máy: tính `F(j)`, `G(j)` **theo định nghĩa** (vét cạn mọi `l` và mọi `k`) rồi so với giá trị DP tại từng vị trí, đồng thời kiểm `G(j) ≥ F(j)`.

In [18]:
def F_def(nums, j):
    return max(sum(nums[l:j + 1]) for l in range(j + 1))

def G_def(nums, j):
    return max(sum(nums[t] ** 2 if t == k else nums[t] for t in range(l, j + 1))
               for l in range(j + 1) for k in range(l, j + 1))

rng = random.Random(99)
for _ in range(200):
    nums = [rng.randint(-7, 7) for _ in range(rng.randint(1, 8))]
    f = g = 0
    for j, x in enumerate(nums):
        f, g = max(f, 0) + x, max(max(f, 0) + x * x, g + x)
        assert f == F_def(nums, j) and g == G_def(nums, j) and g >= f
print("F(j), G(j) của DP khớp định nghĩa tại mọi vị trí (200 mảng ngẫu nhiên)")

F(j), G(j) của DP khớp định nghĩa tại mọi vị trí (200 mảng ngẫu nhiên)


## 11. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn `(i, l, r)` | `O(n³)` | `O(1)` |
| 2. Mỗi `i` một lượt Kadane | `O(n²)` | `O(1)` |
| 3. Kadane trái/phải | `O(n)` | `O(n)` |
| 4. DP `f, g` | `O(n)` | `O(1)` |

Giá trị lớn nhất: `10⁸ + 10⁵·10⁴ ≈ 1.1·10⁹` — vừa `int32` ở ngôn ngữ khác nhưng sát; Python không lo tràn. Đo ở `n = 10⁵`:

In [19]:
rng = random.Random(1746)
big = [rng.randint(-10**4, 10**4) for _ in range(10**5)]
for name, sol in [("Cách 3", SolutionLR()), ("Cách 4", SolutionDP()), ("Nộp", Solution())]:
    t = time.perf_counter()
    r = sol.maxSumAfterOperation(big)
    print(f"{name}: {r} trong {1000*(time.perf_counter()-t):.1f} ms")
worst = [10**4] * 10**5
print("worst-case giá trị:", Solution().maxSumAfterOperation(worst), "< 2^31-1 ?", Solution().maxSumAfterOperation(worst) < 2**31 - 1)

Cách 3: 101679299 trong 27.2 ms
Cách 4: 101679299 trong 10.9 ms
Nộp: 101679299 trong 10.4 ms


worst-case giá trị: 1099990000 < 2^31-1 ? True


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [20]:
sols = [Solution(), SolutionDP(), SolutionLR()]
edge = examples + [
    ([5], 25),                 # một phần tử dương
    ([-7], 49),                # một phần tử âm: bình phương thành dương
    ([0], 0),
    ([0, 0, 0], 0),
    ([-1, -1, -1], 1),         # chỉ lấy một phần tử đã bình phương
    ([1, 1, 1], 3),            # 1² = 1: thao tác không đổi gì
    ([-10**4, 10**4], 10**8 + 10**4),
]
for nums, want in edge:
    for s in sols:
        assert s.maxSumAfterOperation(nums) == want, (nums, s)
    assert max_after_op_brute(nums) == want and max_after_op_each(nums) == want
    print("OK", nums, "->", want)

OK [2, -1, -4, -3] -> 17
OK [1, -1, 1, 1, -1, -1, 1] -> 4
OK [5] -> 25
OK [-7] -> 49
OK [0] -> 0
OK [0, 0, 0] -> 0
OK [-1, -1, -1] -> 1
OK [1, 1, 1] -> 3
OK [-10000, 10000] -> 100010000


### 12.2 Ngẫu nhiên có seed, đối chiếu vét cạn

In [21]:
rng = random.Random(2021)
for trial in range(800):
    n = rng.randint(1, 9)
    lim = rng.choice([3, 10, 10**4])
    nums = [rng.randint(-lim, lim) for _ in range(n)]
    want = max_after_op_brute(nums)
    for s in sols:
        assert s.maxSumAfterOperation(nums) == want, nums
    assert max_after_op_each(nums) == want
for trial in range(20):
    nums = [rng.randint(-50, 50) for _ in range(rng.randint(50, 300))]
    assert len({s.maxSumAfterOperation(nums) for s in sols} | {max_after_op_each(nums)}) == 1
print("800 case nhỏ (vét cạn) + 20 case vừa (Cách 2): khớp")

800 case nhỏ (vét cạn) + 20 case vừa (Cách 2): khớp


### 12.3 Kiểm tra "witness" — đoạn khôi phục được thực sự đạt tổng tối ưu

In [22]:
rng = random.Random(5)
for _ in range(300):
    nums = [rng.randint(-8, 8) for _ in range(rng.randint(1, 10))]
    val, l, r, sq = dp_with_witness(nums)
    assert l <= sq <= r
    seg = [nums[k] * nums[k] if k == sq else nums[k] for k in range(l, r + 1)]
    assert sum(seg) == val == max_after_op_brute(nums)
print("witness đúng trên 300 case")

witness đúng trên 300 case


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên `max(f, 0)` khi bình phương tại chỗ (không cho "bắt đầu mới")

In [23]:
def wrong_no_restart(nums):
    f = g = 0
    ans = -inf
    first = True
    for x in nums:
        if first:
            f, g = x, x * x; first = False
        else:
            f, g = max(f, 0) + x, max(f + x * x, g + x)     # SAI: f có thể âm
        ans = max(ans, g)
    return ans

nums = [-1, -1, 3]
print("sai:", wrong_no_restart(nums), "| đúng:", Solution().maxSumAfterOperation(nums), "| vét cạn:", max_after_op_brute(nums))

sai: 8 | đúng: 9 | vét cạn: 9


**Vì sao sai:** trước số `3`, đoạn chưa-dùng tốt nhất có tổng `-1` (âm). Nối vào chỉ làm giảm: `-1 + 9 = 8`, trong khi bắt đầu mới tại `3²` cho `9`. **Sửa:** dùng `max(f, 0) + x*x`.

### Lỗi 2 — Cập nhật `f` trước rồi dùng `f` mới để tính `g`

In [24]:
def wrong_order(nums):
    f = g = 0
    ans = -inf
    for x in nums:
        f = max(f, 0) + x                       # f đã bao gồm x ...
        g = max(max(f, 0) + x * x, g + x)       # SAI: ... rồi lại cộng x² => x tính 2 lần
        ans = max(ans, g)
    return ans

nums = [1, 5]
print("sai:", wrong_order(nums), "| đúng:", Solution().maxSumAfterOperation(nums))

sai: 31 | đúng: 26


**Vì sao sai:** `f` mới đã chứa `x = 5`, cộng thêm `x² = 25` nghĩa là phần tử 5 xuất hiện hai lần: `1 + 5 + 25 = 31`. **Sửa:** gán song song `f, g = ..., ...` hoặc lưu `pf = max(f, 0)` trước.

### Lỗi 3 — Trả về `g` ở phần tử cuối thay vì max trên mọi vị trí

In [25]:
def wrong_return_last(nums):
    f = g = 0
    for x in nums:
        f, g = max(f, 0) + x, max(max(f, 0) + x * x, g + x)
    return g                                   # SAI: chỉ là đoạn KẾT THÚC ở cuối mảng

nums = [5, -1, -1]
print("sai:", wrong_return_last(nums), "| đúng:", Solution().maxSumAfterOperation(nums))

sai: 23 | đúng: 25


**Vì sao sai:** `g` là tốt nhất **kết thúc tại** vị trí hiện tại; đoạn tối ưu `[25]` kết thúc ở chỉ số 0. **Sửa:** cập nhật `ans = max(ans, g)` mỗi bước.

### Lỗi 4 — Tham lam "bình phương phần tử có trị tuyệt đối lớn nhất"

In [26]:
def wrong_greedy_abs(nums):
    i = max(range(len(nums)), key=lambda k: abs(nums[k]))
    b = nums[:]; b[i] = b[i] ** 2
    return kadane(b)

nums = [-7, -6, -6, -6, 6, 6, 6, 6, 6, 6, 6, 6]
print("tham lam:", wrong_greedy_abs(nums), "| đúng:", Solution().maxSumAfterOperation(nums),
      "| vét cạn:", max_after_op_brute(nums))

tham lam: 79 | đúng: 84 | vét cạn: 84


**Vì sao sai:** `(-7)² = 49` lớn nhất, nhưng `-7` bị ngăn cách khỏi dãy số dương bởi ba số `-6` (mất 18). Bình phương số `-6` **sát** dãy dương: `36 + 8·6 = 84` > `49 - 18 + 48 = 79`. Giá trị bình phương lớn chưa chắc tốt — phải xét **ngữ cảnh** hai bên ⇒ DP. (Ví dụ tự dựng, đã đối chiếu vét cạn.)

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Mảng con tổng lớn nhất + được làm một thao tác đặc biệt"** (xóa 1 phần tử, bình phương 1 phần tử, đổi dấu 1 đoạn…) ⇒ Kadane + **chiều trạng thái "đã dùng/chưa dùng"**.
- Mỗi trạng thái `g` có hai nguồn: **dùng ngay tại đây** (từ `f`) và **đã dùng trước** (từ `g`).
- Nhiều trạng thái cùng cập nhật ⇒ **gán song song** hoặc biến tạm.
- Cách "tiền tố trái + hậu tố phải" là phương án dự phòng khi chưa nghĩ ra DP: `best(i) = L[i-1]⁺ + val(i) + R[i+1]⁺`.
- Đáp án là max **toàn cục** của trạng thái, không phải giá trị cuối.

## 15. Biến thể của bài

1. **LeetCode 1186 — Maximum Subarray Sum with One Deletion:** thao tác là **xóa** tối đa một phần tử: `g = max(f_cũ, g + x)` (xóa `x` thì lấy `f` cũ).
2. **Tối đa `k` lần bình phương:** mở rộng thành `k+1` trạng thái `dp[j]` = đã dùng `j` lần (Bài tập 5).
3. **Thay bằng phép khác:** `nums[i] → -nums[i]` (đổi dấu), `nums[i] → nums[i]·c`… — chỉ đổi biểu thức `x*x`.
4. **Mảng vòng (circular):** kết hợp ý tưởng LeetCode 918 *Maximum Sum Circular Subarray* (tổng − mảng con nhỏ nhất).
5. **Phải bình phương phần tử trong một tập chỉ số cho trước:** tại các chỉ số không được phép, `g` chỉ nhận nguồn `g + x`.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi `(i, l, r)` | `O(n³)` | `O(1)` | chuẩn đối chiếu `n ≤ 50` |
| 2. Kadane mỗi `i` | bình phương rồi Kadane | `O(n²)` | `O(1)` | `n ≤ 3000` |
| 3. Trái + phải | `L⁺ + x² + R⁺` | `O(n)` | `O(n)` | dễ nghĩ, dễ khôi phục đoạn |
| 4. DP `f, g` | hai trạng thái dùng/chưa dùng | `O(n)` | `O(1)` | **nộp bài**, tổng quát hóa tốt |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `maxSumAfterOperation([-2])` và `maxSumAfterOperation([0, -3, 0])` trả về gì?

**Bài 2 (tính tay):** Lập bảng `f, g, best` cho `nums = [3, -4, 2]` theo Cách 4.

**Bài 3 (điền TODO):** Viết `max_after_one_deletion(nums)` (LeetCode 1186): tổng lớn nhất của mảng con khác rỗng sau khi **xóa tối đa một** phần tử (mảng con sau khi xóa vẫn phải khác rỗng).

**Bài 4 (tự viết lại):** Viết lại Cách 3 nhưng trả về luôn bộ `(tổng, l, r, i)` — đoạn tối ưu và chỉ số bình phương.

**Bài 5 (mini challenge):** Cho thêm số `k`: được bình phương **tối đa `k`** phần tử. Viết `max_after_k_ops(nums, k)` chạy `O(n·k)`.

In [27]:
def max_after_one_deletion(nums):
    # TODO: hai trạng thái: chưa xóa / đã xóa
    pass

def lr_with_witness(nums):
    # TODO: Cách 3 + khôi phục (tổng, l, r, i)
    pass

def max_after_k_ops(nums, k):
    # TODO: dp[j] = tốt nhất kết thúc tại đây khi đã dùng j lần
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Bắt buộc bình phương một phần tử | `(-2)² = 4` | `[0, 9, 0]` → 9 |
| 2 | Bắt đầu `f = g = 0` | Tại `-4`: `g = max(3 + 16, 9 - 4)` | Đáp án 21 |
| 3 | `g = max(f_cũ, g + x)` | `f_cũ` là đoạn kết thúc ở phần tử trước — xóa `x` | Khởi tạo `g = -inf` để không "xóa" phần tử duy nhất |
| 4 | Lưu điểm bắt đầu của Kadane trái, điểm kết thúc của Kadane phải | `L > 0` thì lấy đoạn trái | So sánh tuple theo tổng |
| 5 | `dp[0]` là Kadane thường | `ndp[j] = max(dp[j] + x, before + x²)` với `before = max(dp[0], 0)` nếu `j = 1`, ngược lại `dp[j-1]` | Chỉ trạng thái 0 lần mới được "bắt đầu mới"; đáp án = max mọi `dp[j]` qua mọi vị trí |

## 19. Đáp án tham khảo

In [28]:
# Bài 1, 2
print("Bài 1:", Solution().maxSumAfterOperation([-2]), Solution().maxSumAfterOperation([0, -3, 0]))
assert Solution().maxSumAfterOperation([-2]) == 4 and Solution().maxSumAfterOperation([0, -3, 0]) == 9
print("Bài 2:"); r2 = max_after_op_dp_demo([3, -4, 2]); assert r2 == 21 == max_after_op_brute([3, -4, 2])

Bài 1: 4 9
Bài 2:
   x |  f (chưa dùng) |  g (đã dùng) | best
   3 |              3 |            9 | 9
  -4 |             -1 |           19 | 19
   2 |              2 |           21 | 21


In [29]:
def max_after_one_deletion(nums):
    f, g = nums[0], -inf     # sau phần tử đầu: chưa xóa = nums[0]; đã xóa = không hợp lệ
    ans = nums[0]
    for x in nums[1:]:
        f, g = max(f, 0) + x, max(f, g + x)   # g: xóa x (giữ đoạn f cũ) hoặc đã xóa trước đó
        ans = max(ans, f, g)
    return ans

def deletion_brute(nums):
    n = len(nums); best = -inf
    for l in range(n):
        for r in range(l, n):
            seg = nums[l:r + 1]
            best = max(best, sum(seg))
            if len(seg) > 1:
                best = max(best, sum(seg) - min(seg))
    return best

assert max_after_one_deletion([1, -2, 0, 3]) == 4      # ví dụ của LeetCode 1186
assert max_after_one_deletion([1, -2, -2, 3]) == 3
assert max_after_one_deletion([-1, -1, -1, -1]) == -1
rng = random.Random(13)
for _ in range(500):
    nums = [rng.randint(-6, 6) for _ in range(rng.randint(1, 8))]
    assert max_after_one_deletion(nums) == deletion_brute(nums), nums
print("Bài 3: OK")

Bài 3: OK


In [30]:
def lr_with_witness(nums):
    n = len(nums)
    left = [None] * n; cur, st = 0, 0
    for i, x in enumerate(nums):
        if cur <= 0: cur, st = 0, i
        cur += x; left[i] = (cur, st)
    right = [None] * n; cur, en = 0, n - 1
    for i in range(n - 1, -1, -1):
        if cur <= 0: cur, en = 0, i
        cur += nums[i]; right[i] = (cur, en)
    best = None
    for i, x in enumerate(nums):
        L, l = (left[i - 1] if i > 0 and left[i - 1][0] > 0 else (0, i))
        R, r = (right[i + 1] if i + 1 < n and right[i + 1][0] > 0 else (0, i))
        cand = (L + x * x + R, l, r, i)
        if best is None or cand[0] > best[0]:
            best = cand
    return best

rng = random.Random(21)
for _ in range(300):
    nums = [rng.randint(-8, 8) for _ in range(rng.randint(1, 10))]
    v, l, r, i = lr_with_witness(nums)
    assert l <= i <= r and v == max_after_op_brute(nums)
    assert sum(nums[k] ** 2 if k == i else nums[k] for k in range(l, r + 1)) == v
print("Bài 4: OK,", lr_with_witness([2, -1, -4, -3]))

Bài 4: OK, (17, 0, 2, 2)


In [31]:
from itertools import combinations

def max_after_k_ops(nums, k):
    # dp[j]: tổng lớn nhất của mảng con KẾT THÚC tại phần tử hiện tại, đã bình phương đúng j phần tử
    dp = [-inf] * (k + 1)
    ans = -inf
    for x in nums:
        ndp = [-inf] * (k + 1)
        ndp[0] = max(dp[0], 0) + x                       # Kadane thường
        for j in range(1, k + 1):
            before = max(dp[0], 0) if j == 1 else dp[j - 1]   # chỉ trạng thái 0 lần mới được "bắt đầu mới"
            ndp[j] = max(dp[j] + x, before + x * x)
        dp = ndp
        ans = max(ans, max(dp))
    return ans

def k_ops_brute(nums, k):
    n = len(nums); best = -inf
    for l in range(n):
        for r in range(l, n):
            seg = nums[l:r + 1]
            for m in range(0, min(k, len(seg)) + 1):
                for S in combinations(range(len(seg)), m):
                    best = max(best, sum(v * v if t in S else v for t, v in enumerate(seg)))
    return best

rng = random.Random(34)
for _ in range(300):
    nums = [rng.randint(-5, 5) for _ in range(rng.randint(1, 7))]
    k = rng.randint(1, 3)
    assert max_after_k_ops(nums, k) == k_ops_brute(nums, k), (nums, k)
    if k == 1:
        assert max_after_k_ops(nums, 1) == Solution().maxSumAfterOperation(nums)
print("Bài 5 (biến thể tự dựng, đối chiếu vét cạn): OK", max_after_k_ops([2, -1, -4, -3], 2))

Bài 5 (biến thể tự dựng, đối chiếu vét cạn): OK 26


Ghi chú Bài 5: vì `x² ≥ x`, "tối đa `k` lần" và "đúng `k` lần" cho cùng đáp án khi `k ≤ n` (các lần thừa đem dùng ngoài mảng con). Bài 5 là biến thể tự dựng — đối chiếu lại đề gốc nếu bạn gặp phiên bản khác.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
f = g = 0; ans = -inf          # f: chưa dùng thao tác, g: đã dùng
for x in nums:
    f, g = max(f, 0) + x, max(max(f, 0) + x * x, g + x)   # GÁN SONG SONG
    ans = max(ans, g)
return ans
```

**Checklist:**
- [ ] `g` có **hai nguồn**: dùng tại `x` (từ `max(f,0)`) và đã dùng trước (`g + x`)?
- [ ] Gán song song / biến tạm để `g` dùng `f` **cũ**?
- [ ] Cập nhật `ans` ở **mọi** vị trí, không chỉ cuối mảng?
- [ ] Mảng 1 phần tử âm (`[-7]` → 49) chạy đúng?
- [ ] Không dùng tham lam "bình phương số lớn nhất"?
- [ ] `O(n)` thời gian, `O(1)` bộ nhớ.